In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import os

In [2]:
#Comparing PopSyn inputs with PopSyn outputs

#Inputs
per_file_in = "Inputs/ss11pid.csv"
hh_file_in = "Inputs/ss11hid.csv"

per_df_in = pd.read_csv(per_file_in)
hh_df_in = pd.read_csv(hh_file_in)

pers_attributes_in = ['STATE','SEX', 'AGEP', 'PWGTP','ESR', 'FOCCP', 'SCHL']
hh_attributes_in = ['STATE','HINCP', 'NP', 'WGTP', 'WIF']

# Output segments by TAZ group
idaho_list = range(1,2190)
wash_list1 = range(2190, 2753)
wash_list2 = range(2765, 2768)
wash_list3 = range(2816, 2818)
mont_list = range(2753, 2762)
oreg_list = [2762, 2763, 2764, 2804, 2805]
nev_list = [2794, 2795, 2821, 2822]
ut_list = [2812, 2813, 2823, 2824, 2825]
wy_list = [2820,2826,2827]	
internal_list = range (1,2753)
srtc_list = range(2234,2753)
external1 = range (2754,2768)
external2 = range(2821,2828)
bmpo_list1 = range(680,874)
bmpo_list2 = range(1094,1117)
btpo_list = range (1117,1761)
compass_list = range(123,680)
kmpo_list = range (874,1094)
lcvmpo_list1 = range(1,123)
lcvmpo_list2 = range(2190,2234)
grp_list = [list(idaho_list), list(wash_list1)+list(wash_list2)+list(wash_list3), list(oreg_list), list(mont_list), list(wy_list), list(nev_list), list(ut_list), list(internal_list), list(srtc_list), list(external1)+list(external2), list(bmpo_list1)+list(bmpo_list2), list(btpo_list), list(compass_list), list(kmpo_list), list(lcvmpo_list1)+list(lcvmpo_list2)]
groups = ['idaho', 'washington', 'oregon', 'montana', 'wyoming', 'nevada', 'utah', 'internal', 'SRTC', 'external_halo', 'BMPO', 'BTPO', 'COMPASS', 'KMPO', 'LCVMPO']

states = ['16', '30', '32', '41', '49', '53', '56']
state_names = ['Idaho', 'Montana', 'Nevada', 'Oregon', 'Utah', 'Washington', 'Wyoming']

#Outputs
per_file_out = "Outputs/PopSyn0_Per.csv"
hh_file_out = "Outputs/PopSyn0_HH.csv"

pers_attributes_out = ['SEX', 'AGE', 'SCHOOL', 'RLABOR']
hh_attributes_out = ['PERSONS', 'UNITS1', 'AUTOS', 'RHHINC']

C:\Users\james.sullivan\AppData\Local\Temp\ipykernel_8476\3484310525.py:7: DtypeWarning: Columns (0: OC, 1: POVPIP, 2: RC) have mixed types. Specify dtype option on import or set low_memory=False.
  per_df_in = pd.read_csv(per_file_in)
C:\Users\james.sullivan\AppData\Local\Temp\ipykernel_8476\3484310525.py:8: DtypeWarning: Columns (0: BDS, 1: BLD, 2: RMS, 3: KIT, 4: PLM, 5: RESMODE, 6: SRNT, 7: SVAL, 8: FACRP, 9: FAGSP, 10: FBDSP, 11: FBLDP, 12: FCONP, 13: FELEP, 14: FFULP, 15: FGASP, 16: FHFLP, 17: FINSP, 18: FKITP, 19: FMHP, 20: FMRGIP, 21: FMRGP, 22: FMRGTP, 23: FMRGXP, 24: FMVP, 25: FPLMP, 26: FRMSP, 27: FRNTMP, 28: FRNTP, 29: FSMP, 30: FSMXHP, 31: FSMXSP, 32: FTAXP, 33: FTELP, 34: FTENP, 35: FVACSP, 36: FVALP, 37: FVEHP, 38: FWATP) have mixed types. Specify dtype option on import or set low_memory=False.
  hh_df_in = pd.read_csv(hh_file_in)


In [3]:
#Find sum and mean of each relevant Output person attribute for each set of TAZs
file_group = [per_file_out, per_file_in]
pers_attributes = [pers_attributes_out, pers_attributes_in]

for pfile in file_group:
    n = file_group.index(pfile)
    per_filename = Path(pfile).stem
    df = pd.read_csv(pfile)

    #All data
    summary_df = []
    df[pers_attributes[n]] = df[pers_attributes[n]].apply(pd.to_numeric, errors='coerce')
    summary_df = df[pers_attributes[n]].agg(['sum', 'count', 'mean'])
    transposed_df = summary_df.T
    df_final = transposed_df.reset_index().rename(columns={'index': 'Attribute'})
    df_final.to_excel(f'{per_filename}_summary.xlsx', index=False, sheet_name='All')

    #Then groups of TAZs - Idaho, internal, and external-halo
    if n == 0:
        for group in grp_list:

            m = grp_list.index(group)
            group_df = df[df['STDM_TAZ'].isin(group)]
            summary_by_group = group_df[pers_attributes[n]].agg(['sum', 'count', 'mean'])
            transposed_group = summary_by_group.T
            final_by_group = transposed_group.reset_index().rename(columns={'index': 'Attribute'})

            with pd.ExcelWriter(f'{per_filename}_summary.xlsx', engine='openpyxl', mode='a') as writer:
                final_by_group.to_excel(writer, sheet_name=groups[m], index=False)

    else:
        summary_cols = [col for col in pers_attributes[n] if col != 'STATE']
        summary_by_group = df.groupby('STATE')[summary_cols].agg(['sum', 'count', 'mean'])

        weight_field = 'PWGTP'
        for attribute in [col for col in summary_cols if col != weight_field]:
            summary_by_group[(attribute, 'weighted_avg')] = (
                df.groupby('STATE').apply(lambda x: np.average(x[attribute], weights=x[weight_field]))
            )

        transposed_group = summary_by_group.T
        final_by_group = transposed_group.reset_index().rename(columns={'level_0': 'Attribute', 'level_1': 'stat'})

        with pd.ExcelWriter(f'{per_filename}_summary.xlsx', engine='openpyxl', mode='a') as writer:
            final_by_group.to_excel(writer, sheet_name='states', index=False)

C:\Users\james.sullivan\AppData\Local\Temp\ipykernel_8476\1108574727.py:8: DtypeWarning: Columns (0: OC, 1: POVPIP, 2: RC) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(pfile)


In [4]:
#Find sum and mean of each relevant Output household attribute for each set of TAZs
hfile_group = [hh_file_out, hh_file_in]
hh_attributes = [hh_attributes_out, hh_attributes_in]

for hfile in hfile_group:

    n = hfile_group.index(hfile)
    hh_filename = Path(hfile).stem
    df = pd.read_csv(hfile)

    #All data
    df[hh_attributes[n]] = df[hh_attributes[n]].apply(pd.to_numeric, errors='coerce')
    summary_df = df[hh_attributes[n]].agg(['sum', 'count', 'mean'])
    transposed_df = summary_df.T
    df_final = transposed_df.reset_index().rename(columns={'index': 'Attribute'})
    df_final.to_excel(f'{hh_filename}_summary.xlsx', index=False, sheet_name='All')

    #Then groups of TAZs - Idaho, internal, and external-halo
    if n == 0:
        for group in grp_list:

            m = grp_list.index(group)
            group_df = df[df['STDM_TAZ'].isin(group)]
            summary_by_group = group_df[hh_attributes[hfile_group.index(hfile)]].agg(['sum', 'count', 'mean'])
            transposed_group = summary_by_group.T
            final_by_group = transposed_group.reset_index().rename(columns={'index': 'Attribute'})

            with pd.ExcelWriter(f'{hh_filename}_summary.xlsx', engine='openpyxl', mode='a') as writer:
                final_by_group.to_excel(writer, sheet_name=groups[m], index=False)

    else:
        summary_cols = [col for col in hh_attributes[n] if col != 'STATE']
        summary_by_group = df.groupby('STATE')[summary_cols].agg(['sum', 'count', 'mean'])

        weight_field = 'WGTP'
        for attribute in [col for col in summary_cols if col != weight_field]:
            summary_by_group[(attribute, 'weighted_avg')] = (
                df.groupby('STATE').apply(lambda x: np.average(x[attribute], weights=x[weight_field]))
            )

        transposed_group = summary_by_group.T
        final_by_group = transposed_group.reset_index().rename(columns={'level_0': 'Attribute', 'level_1': 'stat'})

        with pd.ExcelWriter(f'{hh_filename}_summary.xlsx', engine='openpyxl', mode='a') as writer:
            final_by_group.to_excel(writer, sheet_name='states', index=False)

C:\Users\james.sullivan\AppData\Local\Temp\ipykernel_8476\1193828731.py:9: DtypeWarning: Columns (0: BDS, 1: BLD, 2: RMS, 3: KIT, 4: PLM, 5: RESMODE, 6: SRNT, 7: SVAL, 8: FACRP, 9: FAGSP, 10: FBDSP, 11: FBLDP, 12: FCONP, 13: FELEP, 14: FFULP, 15: FGASP, 16: FHFLP, 17: FINSP, 18: FKITP, 19: FMHP, 20: FMRGIP, 21: FMRGP, 22: FMRGTP, 23: FMRGXP, 24: FMVP, 25: FPLMP, 26: FRMSP, 27: FRNTMP, 28: FRNTP, 29: FSMP, 30: FSMXHP, 31: FSMXSP, 32: FTAXP, 33: FTELP, 34: FTENP, 35: FVACSP, 36: FVALP, 37: FVEHP, 38: FWATP) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(hfile)


In [5]:
# Build state/group comparison tables for the input and output files

from pathlib import Path

per_df_in = pd.read_csv(per_file_in)
hh_df_in = pd.read_csv(hh_file_in)
per_df_out = pd.read_csv(per_file_out)
hh_df_out = pd.read_csv(hh_file_out)

# Helper utilities

def safe_numeric(series):
    return pd.to_numeric(series, errors='coerce').fillna(0)


def weighted_average(values, weights):
    values = pd.to_numeric(values, errors='coerce')
    weights = pd.to_numeric(weights, errors='coerce').fillna(0)
    mask = weights > 0
    return float(np.average(values[mask], weights=weights[mask])) if mask.any() else 0


# Input person summary by state
input_person_rows = []
for state_code, state_name in zip(states, state_names):
    state_df = per_df_in[per_df_in['STATE'].astype(str) == state_code].copy()
    if state_df.empty:
        continue

    state_df['PWGTP'] = safe_numeric(state_df['PWGTP'])
    state_df['AGEP'] = safe_numeric(state_df['AGEP'])
    state_df['SEX'] = safe_numeric(state_df['SEX'])
    state_df['ESR'] = safe_numeric(state_df['ESR']) if 'ESR' in state_df.columns else 0

    input_person_rows.append({
        'State': state_name,
        'State Code': state_code,
        'Population': int(state_df['PWGTP'].sum()),
        'Avg Age': round(weighted_average(state_df['AGEP'], state_df['PWGTP']), 1),
        'Sex': round(weighted_average(state_df['SEX'], state_df['PWGTP']), 2)-1,
        'Workers': int(state_df.loc[state_df['ESR'].isin([1, 2, 3, 4, 5]), 'PWGTP'].sum()),
    })

input_person_summary = pd.DataFrame(input_person_rows)

# Input household summary by state
input_hh_rows = []
for state_code, state_name in zip(states, state_names):
    state_df = hh_df_in[hh_df_in['STATE'].astype(str) == state_code].copy()
    if state_df.empty:
        continue

    state_df['WGTP'] = safe_numeric(state_df['WGTP'])
    state_df['NP'] = safe_numeric(state_df['NP'])
    state_df['HINCP'] = safe_numeric(state_df['HINCP'])

    input_hh_rows.append({
        'State': state_name,
        'State Code': state_code,
        'Households': int(state_df['WGTP'].sum()),
        'Avg Household Size': round(weighted_average(state_df['NP'], state_df['WGTP']), 2),
        'Avg Household Income': round(weighted_average(state_df['HINCP'], state_df['WGTP']), 0),
    })

input_hh_summary = pd.DataFrame(input_hh_rows)


# Output person summary by zone groups
output_person_rows = []
for group_name, zone_group in zip(groups, grp_list):
    group_df = per_df_out[per_df_out['STDM_TAZ'].isin(zone_group)].copy()
    if group_df.empty:
        continue

    group_df['AGE'] = safe_numeric(group_df['AGE']) if 'AGE' in group_df.columns else 0
    group_df['SEX'] = safe_numeric(group_df['SEX']) if 'SEX' in group_df.columns else 0
    group_df['RLABOR'] = safe_numeric(group_df['RLABOR']) if 'RLABOR' in group_df.columns else 0

    output_person_rows.append({
        'State/Group': group_name,
        'Population': int(len(group_df)),
        'Avg Age': round(group_df['AGE'].mean(), 1),
        'Sex': round(group_df['SEX'].mean(), 2),
        'Workers': int(group_df['RLABOR'].gt(0).sum()),
    })

output_person_summary = pd.DataFrame(output_person_rows)

# Output household summary by zone groups
output_hh_rows = []
for group_name, zone_group in zip(groups, grp_list):
    group_df = hh_df_out[hh_df_out['STDM_TAZ'].isin(zone_group)].copy()
    if group_df.empty:
        continue

    group_df['PERSONS'] = safe_numeric(group_df['PERSONS']) if 'PERSONS' in group_df.columns else 0
    group_df['RHHINC'] = safe_numeric(group_df['RHHINC']) if 'RHHINC' in group_df.columns else 0

    output_hh_rows.append({
        'State/Group': group_name,
        'Households': int(len(group_df)),
        'Avg Household Size': round(group_df['PERSONS'].mean(), 2),
        'Avg Household Income': round(group_df['RHHINC'].mean(), 0),
    })

output_hh_summary = pd.DataFrame(output_hh_rows)


# Comparison tables are retained in memory for the final summary workbook.

C:\Users\james.sullivan\AppData\Local\Temp\ipykernel_8476\2539548490.py:5: DtypeWarning: Columns (0: OC, 1: POVPIP, 2: RC) have mixed types. Specify dtype option on import or set low_memory=False.
  per_df_in = pd.read_csv(per_file_in)
C:\Users\james.sullivan\AppData\Local\Temp\ipykernel_8476\2539548490.py:6: DtypeWarning: Columns (0: BDS, 1: BLD, 2: RMS, 3: KIT, 4: PLM, 5: RESMODE, 6: SRNT, 7: SVAL, 8: FACRP, 9: FAGSP, 10: FBDSP, 11: FBLDP, 12: FCONP, 13: FELEP, 14: FFULP, 15: FGASP, 16: FHFLP, 17: FINSP, 18: FKITP, 19: FMHP, 20: FMRGIP, 21: FMRGP, 22: FMRGTP, 23: FMRGXP, 24: FMVP, 25: FPLMP, 26: FRMSP, 27: FRNTMP, 28: FRNTP, 29: FSMP, 30: FSMXHP, 31: FSMXSP, 32: FTAXP, 33: FTELP, 34: FTENP, 35: FVACSP, 36: FVALP, 37: FVEHP, 38: FWATP) have mixed types. Specify dtype option on import or set low_memory=False.
  hh_df_in = pd.read_csv(hh_file_in)


In [6]:
# Create a final summary sheet with states across the columns and an all-states summary column

state_group_lookup = {
    'idaho': 'Idaho',
    'washington': 'Washington',
    'oregon': 'Oregon',
    'montana': 'Montana',
    'wyoming': 'Wyoming',
    'nevada': 'Nevada',
    'utah': 'Utah',
}

input_person_by_state = input_person_summary.set_index('State').sort_index()
input_hh_by_state = input_hh_summary.set_index('State').sort_index()

output_person_by_state = (
    output_person_summary.assign(State=output_person_summary['State/Group'].map(state_group_lookup))
    .dropna(subset=['State'])
    .set_index('State')
    .sort_index()
)

output_hh_by_state = (
    output_hh_summary.assign(State=output_hh_summary['State/Group'].map(state_group_lookup))
    .dropna(subset=['State'])
    .set_index('State')
    .sort_index()
)

# Summary helpers

def summarize_all_states(state_df, metric, weight_col=None):
    if metric in ['Population', 'Workers', 'Households']:
        return int(state_df[metric].sum())

    if metric in ['Avg Age', 'Avg Household Size', 'Avg Household Income']:
        if weight_col is None:
            return float(state_df[metric].mean())
        weights = state_df[weight_col].astype(float)
        values = state_df[metric].astype(float)
        return float(np.average(values, weights=weights))

    if metric in ['Sex']:
        weights = state_df['Population'].astype(float)
        values = state_df['Sex'].astype(float)
        return float(np.average(values, weights=weights))

    return None

final_rows = [
    {
        'Attribute': 'Input Population',
        **{state: int(input_person_by_state.loc[state, 'Population']) for state in state_names},
        'All States': summarize_all_states(input_person_by_state, 'Population'),
    },
    {
        'Attribute': 'Output Population',
        **{state: int(output_person_by_state.loc[state, 'Population']) for state in state_names},
        'All States': summarize_all_states(output_person_by_state, 'Population'),
    },
    {
        'Attribute': 'Input Avg Age',
        **{state: float(input_person_by_state.loc[state, 'Avg Age']) for state in state_names},
        'All States': summarize_all_states(input_person_by_state, 'Avg Age', 'Population'),
    },
    {
        'Attribute': 'Output Avg Age',
        **{state: float(output_person_by_state.loc[state, 'Avg Age']) for state in state_names},
        'All States': summarize_all_states(output_person_by_state, 'Avg Age', 'Population'),
    },
    {
        'Attribute': 'Input Sex',
        **{state: float(input_person_by_state.loc[state, 'Sex']) for state in state_names},
        'All States': summarize_all_states(input_person_by_state, 'Sex'),
    },
    {
        'Attribute': 'Output Sex',
        **{state: float(output_person_by_state.loc[state, 'Sex']) for state in state_names},
        'All States': summarize_all_states(output_person_by_state, 'Sex'),
    },
    {
        'Attribute': 'Input Workers',
        **{state: int(input_person_by_state.loc[state, 'Workers']) for state in state_names},
        'All States': summarize_all_states(input_person_by_state, 'Workers'),
    },
    {
        'Attribute': 'Output Workers',
        **{state: int(output_person_by_state.loc[state, 'Workers']) for state in state_names},
        'All States': summarize_all_states(output_person_by_state, 'Workers'),
    },
    {
        'Attribute': 'Input Households',
        **{state: int(input_hh_by_state.loc[state, 'Households']) for state in state_names},
        'All States': summarize_all_states(input_hh_by_state, 'Households'),
    },
    {
        'Attribute': 'Output Households',
        **{state: int(output_hh_by_state.loc[state, 'Households']) for state in state_names},
        'All States': summarize_all_states(output_hh_by_state, 'Households'),
    },
    {
        'Attribute': 'Input Avg Household Size',
        **{state: float(input_hh_by_state.loc[state, 'Avg Household Size']) for state in state_names},
        'All States': summarize_all_states(input_hh_by_state, 'Avg Household Size', 'Households'),
    },
    {
        'Attribute': 'Output Avg Household Size',
        **{state: float(output_hh_by_state.loc[state, 'Avg Household Size']) for state in state_names},
        'All States': summarize_all_states(output_hh_by_state, 'Avg Household Size', 'Households'),
    },
    {
        'Attribute': 'Input Avg Household Income',
        **{state: float(input_hh_by_state.loc[state, 'Avg Household Income']) for state in state_names},
        'All States': summarize_all_states(input_hh_by_state, 'Avg Household Income', 'Households'),
    },
    {
        'Attribute': 'Output Avg Household Income',
        **{state: float(output_hh_by_state.loc[state, 'Avg Household Income']) for state in state_names},
        'All States': summarize_all_states(output_hh_by_state, 'Avg Household Income', 'Households'),
    },
]

final_summary = pd.DataFrame(final_rows)
final_summary = final_summary[['Attribute'] + state_names + ['All States']]

final_output_path = Path('State_Comparison_Final.xlsx')
final_output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(final_output_path, engine='openpyxl') as writer:
    final_summary.to_excel(writer, sheet_name='Final_State_Summary', index=False)

final_summary

,Attribute,Idaho,Montana,Nevada,Oregon,Utah,Washington,Wyoming,All States
0,Input Population,1934263.00,854664.00,183977.00,465269.00,447042.00,770581.00,118230.00,4.774026e+06
1,Output Population,1720713.00,343547.00,74382.00,53541.00,226766.00,633180.00,48681.00,3.100810e+06
2,Input Avg Age,38.30,41.10,41.30,41.50,34.30,39.70,41.90,3.916931e+01
3,Output Avg Age,37.70,40.50,41.60,40.80,35.00,38.70,42.40,3.823783e+01
4,Input Sex,0.50,0.49,0.48,0.49,0.49,0.50,0.49,4.952804e-01
5,Output Sex,0.50,0.51,0.51,0.52,0.50,0.51,0.51,5.038921e-01
6,Input Workers,960240.00,438469.00,81134.00,212355.00,234525.00,374543.00,62087.00,2.363353e+06
7,Output Workers,1319572.00,283998.00,59806.00,42810.00,171093.00,509664.00,39772.00,2.426715e+06
8,Input Households,795014.00,409106.00,81856.00,213120.00,172140.00,336929.00,59444.00,2.067609e+06
9,Output Households,718594.00,170038.00,35352.00,26080.00,98956.00,261232.00,23666.00,1.333918e+06
